# Airline Operational Performance Analytics

### Project Overview

Feature engineering is the process of creating new variables from existing data to improve analysis and support future predictive modeling. These engineered features provide more meaningful business metrics than the raw dataset.

This notebook focuses on creating operational performance indicators that help evaluate airline efficiency, delay severity, and seasonal travel patterns.

## Table of Contents

1. Import Libraries
2. Load Cleaned Dataset
3. Dataset Overview
4. Feature 1: Delay Rate (%)
5. Feature 2: Average Delay per Flight
6. Feature 3: On-Time Flights
7. Feature 4: On-Time Percentage
8. Feature 5: Total Delay Count
9. Feature 6: Delay Severity Category
10. Feature 7: Quarter
11. Feature 8: Season
12. Feature 9: Peak Travel Indicator
13. Feature Summary
14. Export Engineered Dataset
15. Conclusion

# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

# 2. Load Cleaned Dataset

In [7]:
df = pd.read_csv("../cleaned_data/airline_delay_cleaned.csv")

df.head()

,year,month,carrier,carrier_name,airport,airport_name,total_arrival_flights,flights_delayed_over_15min,carrier_delay_count,weather_delay_count,...,security_delay_count,late_aircraft_delay_count,cancelled_flights,diverted_flights,total_arrival_delay_minutes,carrier_delay_minutes,weather_delay_minutes,nas_delay_minutes,security_delay_minutes,late_aircraft_delay_minutes
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30,0,0.00,0.0,...,0.0,0.00,0,0,0,0,0,0,0,0
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2,0,0.00,0.0,...,0.0,0.00,0,0,0,0,0,0,0,0
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28,8,3.74,0.0,...,0.0,2.66,2,0,409,236,0,70,0,103
3,2025,1,G4,Allegiant Air,EVV,"Evansville, IN: Evansville Regional",18,1,0.00,1.0,...,0.0,0.00,0,0,1075,0,1075,0,0,0
4,2025,1,G4,Allegiant Air,EWR,"Newark, NJ: Newark Liberty International",31,5,2.17,0.0,...,0.0,0.00,1,0,446,336,0,110,0,0


# 3. Dataset Overview

Before creating new features, let's review the cleaned dataset to ensure it has been loaded correctly.

In [9]:
print(f"Rows : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

df.info()

Rows : 397283
Columns : 21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397283 entries, 0 to 397282
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   year                         397283 non-null  int64  
 1   month                        397283 non-null  int64  
 2   carrier                      397283 non-null  object 
 3   carrier_name                 397283 non-null  object 
 4   airport                      397283 non-null  object 
 5   airport_name                 397283 non-null  object 
 6   total_arrival_flights        397283 non-null  int64  
 7   flights_delayed_over_15min   397283 non-null  int64  
 8   carrier_delay_count          397283 non-null  float64
 9   weather_delay_count          397283 non-null  float64
 10  nas_delay_count              397283 non-null  float64
 11  security_delay_count         397283 non-null  float64
 12  late_aircraft_delay_count    39

# 4. Feature 1: Delay Rate (%)

## Why create this feature?

The raw dataset provides the total number of delayed flights and total arrival flights separately. Calculating the delay rate gives a standardized metric that measures the percentage of flights delayed, allowing fair comparisons across airlines and airports of different sizes.

In [10]:
df["delay_rate_percentage"] = (
    df["flights_delayed_over_15min"] /
    df["total_arrival_flights"]
) * 100

df["delay_rate_percentage"] = df["delay_rate_percentage"].round(2)

df[[
    "total_arrival_flights",
    "flights_delayed_over_15min",
    "delay_rate_percentage"
]].head()

,total_arrival_flights,flights_delayed_over_15min,delay_rate_percentage
0,30,0,0.00
1,2,0,0.00
2,28,8,28.57
3,18,1,5.56
4,31,5,16.13


### Observation

The newly created **delay_rate_percentage** feature represents the proportion of delayed flights for each airport-airline combination. Unlike total delay counts, this percentage enables fair performance comparisons regardless of flight volume.

# 5. Feature 2: Average Delay per Flight

## Why create this feature?

The total arrival delay minutes indicate the cumulative delay experienced by all flights. However, this metric alone does not account for differences in flight volume. Calculating the average delay per flight provides a standardized measure of operational efficiency and enables fair comparisons across airlines and airports.

In [11]:
df["average_delay_per_flight"] = (
    df["total_arrival_delay_minutes"] /
    df["total_arrival_flights"]
)

df["average_delay_per_flight"] = df["average_delay_per_flight"].round(2)

df[[
    "total_arrival_flights",
    "total_arrival_delay_minutes",
    "average_delay_per_flight"
]].head()

,total_arrival_flights,total_arrival_delay_minutes,average_delay_per_flight
0,30,0,0.00
1,2,0,0.00
2,28,409,14.61
3,18,1075,59.72
4,31,446,14.39


### Observation

The **average_delay_per_flight** feature measures the average number of delay minutes experienced per arriving flight. Unlike total delay minutes, this feature normalizes delay performance by flight volume, making it a more reliable indicator for comparing operational efficiency.

# 6. Feature 3: On-Time Flights

## Why create this feature?

The dataset records the total number of arriving flights and the number of flights delayed by more than 15 minutes. Calculating on-time flights helps quantify operational reliability by identifying how many flights arrived without significant delays.

In [12]:
df["on_time_flights"] = (
    df["total_arrival_flights"] -
    df["flights_delayed_over_15min"]
)

df[[
    "total_arrival_flights",
    "flights_delayed_over_15min",
    "on_time_flights"
]].head()

,total_arrival_flights,flights_delayed_over_15min,on_time_flights
0,30,0,30
1,2,0,2
2,28,8,20
3,18,1,17
4,31,5,26


### Observation

The **on_time_flights** feature represents the number of flights that arrived without experiencing delays greater than 15 minutes. This metric provides a direct measure of operational reliability and supports the calculation of on-time performance percentages.

# 7. Feature 4: On-Time Percentage

## Why create this feature?

While the number of on-time flights provides useful information, converting it into a percentage enables standardized performance comparisons across airports and airlines regardless of their traffic volume.

In [13]:
df["on_time_percentage"] = (
    df["on_time_flights"] /
    df["total_arrival_flights"]
) * 100

df["on_time_percentage"] = df["on_time_percentage"].round(2)

df[[
    "on_time_flights",
    "total_arrival_flights",
    "on_time_percentage"
]].head()

,on_time_flights,total_arrival_flights,on_time_percentage
0,30,30,100.00
1,2,2,100.00
2,20,28,71.43
3,17,18,94.44
4,26,31,83.87


### Observation

The **on_time_percentage** feature measures the proportion of flights that arrived on time. This standardized metric provides a clear indicator of airline and airport operational performance and facilitates meaningful comparisons across different traffic volumes.

# 8. Feature 5: Total Delay Count

## Why create this feature?

The dataset stores delay counts separately for each delay category. Combining these values into a single feature provides the total number of delay incidents, making it easier to evaluate the overall operational disruption experienced by flights.

In [14]:
df["total_delay_count"] = (
    df["carrier_delay_count"] +
    df["weather_delay_count"] +
    df["nas_delay_count"] +
    df["security_delay_count"] +
    df["late_aircraft_delay_count"]
)

df[[
    "carrier_delay_count",
    "weather_delay_count",
    "nas_delay_count",
    "security_delay_count",
    "late_aircraft_delay_count",
    "total_delay_count"
]].head()

,carrier_delay_count,weather_delay_count,nas_delay_count,security_delay_count,late_aircraft_delay_count,total_delay_count
0,0.00,0.0,0.00,0.0,0.00,0.0
1,0.00,0.0,0.00,0.0,0.00,0.0
2,3.74,0.0,1.60,0.0,2.66,8.0
3,0.00,1.0,0.00,0.0,0.00,1.0
4,2.17,0.0,2.83,0.0,0.00,5.0


### Observation

The **total_delay_count** feature combines all individual delay categories into a single operational metric, providing a comprehensive view of the number of delay events affecting flights.

# 9. Feature 6: Delay Severity Category

## Why create this feature?

Delay rate is a continuous numerical value. Categorizing it into severity levels simplifies interpretation and enables easier reporting, filtering, and future predictive analysis.

In [15]:
df["delay_severity"] = pd.cut(
    df["delay_rate_percentage"],
    bins=[0, 20, 40, 60, 80, 100],
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ],
    include_lowest=True
)

df[[
    "delay_rate_percentage",
    "delay_severity"
]].head()

,delay_rate_percentage,delay_severity
0,0.00,Very Low
1,0.00,Very Low
2,28.57,Low
3,5.56,Very Low
4,16.13,Very Low


### Observation

The **delay_severity** feature classifies operational performance into five categories based on delay rate, making performance assessment more intuitive than interpreting raw percentages.

# 10. Feature 7: Quarter

## Why create this feature?

Grouping months into quarters enables quarterly performance analysis and supports seasonal business reporting.

In [16]:
quarter_mapping = {
    1: "Q1",
    2: "Q1",
    3: "Q1",
    4: "Q2",
    5: "Q2",
    6: "Q2",
    7: "Q3",
    8: "Q3",
    9: "Q3",
    10: "Q4",
    11: "Q4",
    12: "Q4"
}

df["quarter"] = df["month"].map(quarter_mapping)

df[[
    "month",
    "quarter"
]].head()

,month,quarter
0,1,Q1
1,1,Q1
2,1,Q1
3,1,Q1
4,1,Q1


### Observation

The **quarter** feature groups monthly records into four business quarters, enabling higher-level trend analysis and quarterly reporting.

# 11. Feature 8: Season

## Why create this feature?

Grouping months into seasons helps analyze how weather and travel demand influence airline performance throughout the year.

In [17]:
season_mapping = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn"
}

df["season"] = df["month"].map(season_mapping)

df[[
    "month",
    "season"
]].head()

,month,season
0,1,Winter
1,1,Winter
2,1,Winter
3,1,Winter
4,1,Winter


### Observation

The **season** feature enables seasonal analysis of airline operations, helping identify patterns associated with weather conditions and travel demand.

# 12. Feature 9: Peak Travel Indicator

## Why create this feature?

Peak travel periods generally experience higher passenger demand and operational pressure. Identifying these months helps compare airline performance during busy and non-busy travel seasons.

In [18]:
peak_months = [6, 7, 8, 11, 12]

df["peak_travel"] = np.where(
    df["month"].isin(peak_months),
    "Peak",
    "Non-Peak"
)

df[[
    "month",
    "peak_travel"
]].head()

,month,peak_travel
0,1,Non-Peak
1,1,Non-Peak
2,1,Non-Peak
3,1,Non-Peak
4,1,Non-Peak


### Observation

The **peak_travel** feature classifies records into peak and non-peak travel periods, supporting comparative analysis of operational performance during high-demand seasons.

# 13. Feature Summary

The following engineered features were created in this notebook:

- Delay Rate Percentage
- Average Delay per Flight
- On-Time Flights
- On-Time Percentage
- Total Delay Count
- Delay Severity
- Quarter
- Season
- Peak Travel Indicator

These engineered features improve data interpretation and provide meaningful business metrics for future analysis and predictive modeling.

In [21]:
print(f"Final Dataset Shape: {df.shape}")

Final Dataset Shape: (397283, 30)


# 14. Export Engineered Dataset

In [19]:
df.to_csv("../cleaned_data/airline_delay_engineered.csv", index=False)

print("Engineered dataset exported successfully.")

Engineered dataset exported successfully.


In [20]:
engineered_columns = [
    "delay_rate_percentage",
    "average_delay_per_flight",
    "on_time_flights",
    "on_time_percentage",
    "total_delay_count",
    "delay_severity",
    "quarter",
    "season",
    "peak_travel"
]

df[engineered_columns].head()

,delay_rate_percentage,average_delay_per_flight,on_time_flights,on_time_percentage,total_delay_count,delay_severity,quarter,season,peak_travel
0,0.00,0.00,30,100.00,0.0,Very Low,Q1,Winter,Non-Peak
1,0.00,0.00,2,100.00,0.0,Very Low,Q1,Winter,Non-Peak
2,28.57,14.61,20,71.43,8.0,Low,Q1,Winter,Non-Peak
3,5.56,59.72,17,94.44,1.0,Very Low,Q1,Winter,Non-Peak
4,16.13,14.39,26,83.87,5.0,Very Low,Q1,Winter,Non-Peak


# 15. Conclusion

This notebook created several meaningful features from the cleaned airline dataset. These engineered features transform raw operational data into standardized business metrics that improve performance analysis and support future predictive modeling tasks.